# Constrained creative writing in English — Kaggle runner (2× T4)WritingPrompts + four verifiable constraints, comparing plain generation againstorthogonal constraint steering.Every story is checkpointed as it is written. If this notebook stops, re-run thesame cell and it continues where it left off.Accelerator must be **GPU T4 x2**.

In [ ]:
# One model per subprocess: several 8B models will not fit in one process on 2x T4.MODELS   = ["Qwen3-8B", "Qwen2.5-7B", "OLMo-2-7B", "Falcon3-7B", "Granite-3.1-8B"]SUITE    = "noise"             # noise   = baseline + the four steering variants                               # compare = baseline vs our method only                               # alpha | beta | core | ortho | loo | allPROMPTS  = 10                  # WritingPrompts promptsSTORIES  = 10                  # stories per prompt -> 100 per condition# These go into the prompt text, so a run refuses to reuse a checkpoint written# under different values. Change them and you need a fresh OUT directory.MAX_WORDS = 60MAX_GRADE = 3.0# Multiples of each model's own measured activation scale.ALPHA    = 0.175               # noise strength   (0 = no noise)BETA     = 1.0                 # steering strengthOUT      = "/kaggle/working/english"REPO     = "/kaggle/working/NoiseEGRA"BRANCH   = "english-generalization"MODELS_STR = " ".join(MODELS)   # used by the shell command below

## 1. Install and clone

In [ ]:
!pip install -q -U "transformers>=4.56" accelerate hf_transfer datasets spacy!python -m spacy download -q en_core_web_sm!rm -rf {REPO} && git clone -q --branch {BRANCH} --single-branch \    https://github.com/haziq-exe/NoiseEGRA.git {REPO}import osos.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # HF's default downloader stalls on large shards!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader# Kaggle wipes the model cache between sessions even when /kaggle/working is# restored, so a resumed run re-downloads the weights. Check space and what is# already cached before starting.!df -h /root /kaggle/working | awk 'NR==1 || /root|working/'!du -sh /root/.cache/huggingface 2>/dev/null || echo "no model cache yet"

## 2. Hugging Face loginQwen3-8B is Apache 2.0 and needs no token. Run this only for Llama-3.1-8B, which isgated: accept the licence on its model page, then add your token under**Add-ons → Secrets** as `HF_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClientfrom huggingface_hub import loginlogin(UserSecretsClient().get_secret("HF_TOKEN"))

## 3. Continuing from an earlier session (skip on the first run)Kaggle wipes `/kaggle/working` when a session ends. **Save Version** before closing,then next session add that version's output under **Add-ons → Add data → Your Work**and put its path here.

In [ ]:
import os, shutil, jsonPREVIOUS = ""   # e.g. "/kaggle/input/english-run-1/english"if PREVIOUS and os.path.isdir(PREVIOUS):    shutil.copytree(PREVIOUS, OUT, dirs_exist_ok=True)    sp = f"{OUT}/{MODEL}/state.json"    n = sum(len(v) for v in json.load(open(sp))["runs"].values()) if os.path.isfile(sp) else 0    print(f"restored {n} stories")else:    print("starting fresh")

## 4. GenerateThe first run downloads the model (~16 GB) and the WritingPrompts dataset, thenextracts the steering vectors once. After that it prints one line per story with arunning estimate of time remaining. Safe to interrupt.

In [ ]:
!cd {REPO} && python -u scripts/run_english_sweep.py \    --models {MODELS_STR} --suite {SUITE} \    --num-prompts {PROMPTS} --stories-per-prompt {STORIES} \    --max-words {MAX_WORDS} --max-grade {MAX_GRADE} \    --alpha {ALPHA} --beta {BETA} \    --out {OUT}

## 5. ScoreConstraint adherence uses the exact checks. `--diversity` adds Vendi and Self-BLEU,computed within each prompt group and then averaged.

In [ ]:
for m in MODELS:    print("\n" + "#" * 100)    print("#", m)    print("#" * 100)    !cd {REPO} && python -u scripts/score_english.py \        --input-dir {OUT}/{m} --diversity \        --max-words {MAX_WORDS} --max-grade {MAX_GRADE}

## 6. Read some storiesNumbers cannot tell you the prose is any good. Read a few before trusting the table.

In [ ]:
import csv, glob, textwrapREAD = MODELS[0]   # which model to eyeballfor path in sorted(glob.glob(f"{OUT}/{READ}/*.csv")):    rows = list(csv.DictReader(open(path, encoding="utf-8")))    print("=" * 100); print(path.split("/")[-1]); print("=" * 100)    for r in rows[:2]:        print(f"\n-- prompt {r['prompt_index']}, story {r['story_index']} --")        print(textwrap.fill(r["story"], 96))    print()